In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

USERNAME = "rishika2"
PASSWORD = "M.ieJ7^tet&Kyc9"

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get("https://www.ravelry.com/account/login")

wait = WebDriverWait(driver, 15)

username_box = wait.until(EC.presence_of_element_located((By.ID, "user_login")))
password_box = driver.find_element(By.ID, "user_password")

username_box.send_keys(USERNAME)
password_box.send_keys(PASSWORD)

login_button = driver.find_element(By.CSS_SELECTOR, "button[type='submit']")
login_button.click()

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://herrschners.com/herrschners/knit-crochet/books-patterns/free-pattern-downloads/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

patterns = []

# product cards
cards = soup.select("a.product-item-link")

for card in cards:
    name = card.get_text(strip=True)
    link = card["href"]

    patterns.append({
        "pattern_name": name,
        "url": link
    })

df = pd.DataFrame(patterns)

print(df.head())

Empty DataFrame
Columns: []
Index: []


In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

BASE_URL = "https://herrschners.com/herrschners/knit-crochet/books-patterns/free-pattern-downloads/"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(BASE_URL, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

rows = []

cards = soup.select("article.card")

for card in cards:

    product_link = card.get("data-href")

    img = card.select_one("img")
    image = img["src"] if img else None
    title = img.get("title") if img else None

    # visit product page
    r = requests.get(product_link, headers=headers)
    product_soup = BeautifulSoup(r.text, "html.parser")

    # scrape INFO section
    info_section = product_soup.select_one(".productView-properties")

    description = info_section.get_text(" ", strip=True) if info_section else None

    rows.append({
        "title": title,
        "product_link": product_link,
        "image": image,
        "description": description
    })

    time.sleep(1)

df = pd.DataFrame(rows)

df.head()

,title,product_link,image,description
0,Village Yarn Blue Burst Pot Holders Free Download,https://herrschners.com/village-yarn-blue-burs...,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ste...,Info SKU:F01729 Info SKU: F01729 UPC: Project ...
1,Willow Yarns Garden Song Shawlette Free Download,https://herrschners.com/willow-yarns-garden-so...,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ste...,Info SKU:F01728 Info SKU: F01728 UPC: Project ...
2,Herrschners English Garden Doily Free Download,https://herrschners.com/herrschners-english-ga...,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ste...,Info SKU:F01727 Info SKU: F01727 UPC: Project ...
3,Herrschners Shamrock Glow Blanket Free Download,https://herrschners.com/herrschners-shamrock-g...,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ste...,Info SKU:F01726 Info SKU: F01726 UPC: Occasion...
4,Herrschners Tilework Bottle Carriers - Set of ...,https://herrschners.com/herrschners-tilework-b...,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ste...,Info SKU:F01725 Info SKU: F01725 UPC: Project ...


In [10]:
df.loc[0, "description"]

'Info SKU:F01729 Info SKU: F01729 UPC: Project Type: Pot Holders Skill: Crochet Difficulty: Easy'

In [ ]:
import requests
import re
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0"}

def scrape_product(product_link):

    r = requests.get(product_link, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    row = {}

    # Getting the info (which contains project type, craft type, and difficulty level, but not always)
    info_section = soup.select_one(".productView-info")
    if info_section:
        row["info"] = desc.get_text(" ", strip=True)

    # Getting description, skill level, size, materials, etc. from the description section)
    desc = soup.select_one(".productView-desc-content")
    if desc:
        row["description"] = desc.get_text(" ", strip=True)

    # Getting review count by extracting the number from the review link text
    review = soup.select_one(".productView-reviewLink")
    # print (review.get_text() if review else "No review link found")

    if review:
        match = re.search(r"\d+", review.get_text())
        row["review_count"] = int(match.group()) if match else 0
    else:
        # if text is no reviews yet
        row["review_count"] = 0

    # Counting number of stars by counting the number of full star icons
    rating_div = soup.select_one(".productView-rating")

    if rating_div:
        stars = rating_div.select(".icon.icon--ratingFull")
        row["star_rating"] = len(stars)
    else:
        row["star_rating"] = 0

    return row

In [16]:
import pandas as pd
import time

URL = "https://herrschners.com/herrschners/knit-crochet/books-patterns/free-pattern-downloads/"

response = requests.get(URL, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

rows = []

cards = soup.select("article.card")

for card in cards:

    product_link = card.get("data-href")
    img = card.select_one("img")
    row = {
        "title": img.get("title") if img else None,
        "image": img["src"] if img else None,
        "product_link": product_link
    }
    features = scrape_product(product_link)
    row.update(features)

    rows.append(row)

    time.sleep(1)

df = pd.DataFrame(rows)

df.head()

,title,image,product_link,sku,upc,project_type,skill,difficulty,description,review_count,star_rating,occasion,season,theme
0,Village Yarn Blue Burst Pot Holders Free Download,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ste...,https://herrschners.com/village-yarn-blue-burs...,F01729,,Pot Holders,Crochet,Easy,Brighten your kitchen with a splash of color! ...,0,0,NaN,NaN,NaN
1,Willow Yarns Garden Song Shawlette Free Download,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ste...,https://herrschners.com/willow-yarns-garden-so...,F01728,,Shawl,Knit,Easy,Be ready for backyard garden parties with this...,0,0,NaN,NaN,NaN
2,Herrschners English Garden Doily Free Download,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ste...,https://herrschners.com/herrschners-english-ga...,F01727,,Doily,Crochet,Easy,No green thumb is needed to grow a garden of t...,0,0,NaN,NaN,NaN
3,Herrschners Shamrock Glow Blanket Free Download,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ste...,https://herrschners.com/herrschners-shamrock-g...,F01726,,Blanket,Crochet,Easy,It’s your lucky day! This free easy crochet bl...,0,0,St. Patrick's Day,NaN,NaN
4,Herrschners Tilework Bottle Carriers - Set of ...,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ste...,https://herrschners.com/herrschners-tilework-b...,F01725,,Fashion Accessories & Novelties,Crochet,Intermediate,Stay hydrated on the go with these cute and co...,1,5,NaN,NaN,NaN


In [17]:
df.shape

(48, 14)